# Validation notebook template
### by Veronica Bossio Botero, 03-28-2025

This notebook investigates the sensitivity and robustness of INSERT MEASURE TO VALIDATE to variations in reading difficulty and text length.

Using the CLEAR Corpus, which includes English-language excerpts tagged with Flesch-Kincaid Grade Level scores, we categorized passages into five readability groups and analyzed whether INSERT MEASURE can accurately reflect increasing text complexity.

After computing the metrics using the OpenWillis pipeline, we conducted:

- **ANOVAs** to assess whether metric distributions vary significantly across grade levels
- **Post-hoc Dunn's tests** to localize where the differences lie
- **Linear regression** to track continuous trends over grade level

To assess dependecy on text length, we "augmented" the CLEAR corpus by sampling substrings of varying lengths for a subset of the excerpts--which yielded CLEAR corpus passages of different lengths--and evaluated whether and how INSERT MEASURE changes as a function of sample length. 

In [ ]:
%reload_ext autoreload
%autoreload 2

import openwillis.speech as ows
print(ows.__file__)

import pandas as pd
import os
import numpy as np
import math
from scipy import stats
import scikit_posthocs as sp
import statsmodels.api as sm

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'
pio.templates.default = 'simple_white'

import random
import nltk
from nltk.tokenize import word_tokenize
from tqdm import tqdm


import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings(action='ignore', category=UserWarning, module='nltk')
warnings.filterwarnings(action='ignore', category=UserWarning, module='tensorflow')

import logging
logging.getLogger('VoskAPI').setLevel(logging.CRITICAL)

# original_stdout = sys.stdout
# sys.stdout = open(os.devnull, 'w')

nltk.download('punkt',quiet=True)
nltk.download('averaged_perceptron_tagger',quiet=True)
nltk.download('wordnet',quiet=True)

import stats_visualizations as sv

### Load Augmented CLEAR DataFrame

We load a pre-computed version of the CLEAR corpus that has been **augmented with truncated variants** of the original excerpts.

These augmented excerpts were generated by sampling random-length substrings from each original passage to simulate varying text lengths. For each original excerpt, multiple truncated versions were created with lengths uniformly drawn from a specified range (e.g., 5 to 200 words). This allows us to analyze how metric values vary as a function of excerpt length, independent of content.

The augmentation was performed using a standalone script (`augment_CLEAR_w_diff_lengths.py`) and saved as a CSV for efficient reuse in this and other validation notebooks.

In [ ]:
PATH_TO_CLEAR = '/Users/veronicabossio/Library/Mobile Documents/com~apple~CloudDocs/brooklyn_health/data/CLEAR/augmented_CLEAR.csv'

### Function to compute your measure

In [ ]:
def compute_your_measures(df):
    """
    Compute lexical measures for each excerpt in the DataFrame.
    """    
    
    your_measure_vals = []
    your_measure_name = 'your_measure'
    
    for json_obj in tqdm(df['json_output']):
        aug_summ_df = ows.speech_characteristics(json_obj, option='simple')[2]
        your_measure_vals.append(aug_summ_df[your_measure_name].values[0])

    # Assign all at once
    aug_new_df = df.copy()
    aug_new_df['your_measure'] = your_measure_vals

    return aug_new_df


In [ ]:
df = pd.read_csv(PATH_TO_CLEAR)

df_w_measures = compute_your_measures(df)

## Measures vs. ease of readability and grade level

In [ ]:
ordinal_map = {'Kindergarten': 0, 'Elementary': 1, 'Teen': 2, 'College': 3, 'Adult': 4}
df_w_measures['Grade_Ordinal'] = df_w_measures['Grade Level'].map(ordinal_map)

In [ ]:
metrics = ['your_measure1', 'your_measure2']

In [ ]:
# get the original exceprts before the length augmentation (full length)
og_df = df_w_measures[df_w_measures['Truncated'] == False]
grouped = og_df.groupby('Grade Level')

### Continuous grade level

In [ ]:
sv.plot_regression_subplots(metrics, og_df, variable='Flesch-Kincaid-Grade-Level', cols=2)

short description of results

### Binned grade level

In [ ]:

grades = ['Kindergarten', 'Elementary', 'Teen', 'College', 'Adult']

# Plotting the regression subplots for your measures
fig = sv.plot_regression_subplots(metrics, og_df, variable='Grade_Ordinal', cols=3)

fig.update_xaxes(
    tickvals=[0, 1, 2, 3, 4],
    ticktext=grades,
    title=''
)
fig

short description of results

## ANOVA for grade level group

In [ ]:
grouped = og_df.groupby('Grade Level')
# Collect results
anova_results = {}

for metric in metrics:
    # Extract metric values for each grade level in order
    groups = [grouped.get_group(level)[metric].dropna() for level in grades] # list of series

    # Run one-way ANOVA and round
    stat, pval = np.round(stats.f_oneway(*groups), 2)
    
    # Store results
    anova_results[metric] = {'F-statistic': stat, 'p-value': pval}

In [ ]:
def make_box_trace(df, metric, grade_order, stat=None, pval=None):
    """
    Returns a list of go.Box traces for each grade level.
    """
    traces = []
    for grade in grade_order:
        traces.append(go.Box(
            y=df[df['Grade Level'] == grade][metric],
            name=grade,
            boxpoints='outliers',
            marker=dict(opacity=0.5, color='royalblue'),
            line=dict(width=1),
            showlegend=False
        ))
    return traces

In [ ]:
cols = 2
rows = math.ceil(len(metrics) / cols)

# Create subplot grid
fig = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[
        f"{metric.replace('_', ' ').title()}<br>F={anova_results[metric]['F-statistic']:.2f}, "
        f"p={anova_results[metric]['p-value']:.3f}" for metric in metrics
    ]
)

for i, metric in enumerate(metrics):
    row = (i // cols) + 1
    col = (i % cols) + 1
    
    traces = make_box_trace(og_df, metric, grades, 
                            stat=anova_results[metric]['F-statistic'], 
                            pval=anova_results[metric]['p-value'])
    
    for trace in traces:
        fig.add_trace(trace, row=row, col=col)
    
    fig.update_yaxes(title_text=metric.replace('_', ' ').title(), row=row, col=col)
    #fig.update_xaxes(title_text="Grade Level", row=row, col=col)

    fig.update_layout(
    height=rows * 400,
    width=cols * 450,
    title_text="Boxplots of Metrics by Grade Level with ANOVA Results",
    showlegend=False,
    template='plotly_white'
)

fig.show()

short description of results

## Post-hoc Dunn tests and effect size

In [ ]:
for metric in metrics:
    fig = sv.plot_posthoc_heatmap(og_df, metric)
    fig.show()

Cohen’s *d* is a standardized measure of **effect size** — it quantifies **how much two groups differ**, relative to their pooled variability. In this context, it shows how strongly each lexical diversity metric separates different readability levels.
Above are the effect sizes for the comparison of each pair of readability-level categories. Stars indicate **statistical significance** (p < 0.05).

short description of results

## Text-sample length dependence

In [ ]:
for metric in metrics:
    fig = sv.plot_length_dependence_by_grade(df_w_measures, metric)
    fig.show()

short description of results